# Customer Churn Prediction and Analysis

## Data Preprocessing

### Objective

The objective of this notebook is to clean and prepare the dataset by handling data quality issues, correcting data types, removing invalid records, and preparing the data for exploratory data analysis and machine learning.

In [3]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

plt.style.use("ggplot")

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Section 1: Data Type Issues

## Problem

During the data understanding phase, we identified that the `TotalCharges` column is stored as a string (`object`) instead of a numeric data type.

This issue prevents numerical calculations and machine learning algorithms from processing the feature correctly.

In [5]:
# Check current data type
print("Current Data Type:")
print(df["TotalCharges"].dtype)

# Count blank values
blank_values = (df["TotalCharges"] == " ").sum()

print(f"\nBlank Values: {blank_values}")

Current Data Type:
str

Blank Values: 11


In [6]:
# Display affected rows

df.loc[df["TotalCharges"] == " ",
       ["customerID",
        "tenure",
        "MonthlyCharges",
        "TotalCharges"]]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


In [7]:
# Replace blank strings with NaN

df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)

# Convert to numeric

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])

# Remove missing records

df.dropna(subset=["TotalCharges"], inplace=True)

In [8]:
print("Current Shape:", df.shape)

print("\nMissing Values:")
print(df["TotalCharges"].isnull().sum())

print("\nCurrent Data Type:")
print(df["TotalCharges"].dtype)

Current Shape: (7032, 21)

Missing Values:
0

Current Data Type:
float64


In [9]:
cleaning_log = pd.DataFrame({
    "Issue": [
        "TotalCharges Data Type",
        "Blank Values",
        "Dataset Rows"
    ],
    
    "Before": [
        "object",
        11,
        7043
    ],
    
    "After": [
        "float64",
        0,
        df.shape[0]
    ],
    
    "Status": [
        "✅ Fixed",
        "✅ Fixed",
        "Updated"
    ]
})

cleaning_log

,Issue,Before,After,Status
0,TotalCharges Data Type,object,float64,✅ Fixed
1,Blank Values,11,0,✅ Fixed
2,Dataset Rows,7043,7032,Updated


# Section 2: Duplicate Records

## Problem

Duplicate records may introduce bias during data analysis and model training.

Therefore, duplicate observations should always be checked before proceeding.

In [10]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate Rows : {duplicate_rows}")

Duplicate Rows : 0


## Decision

No duplicate records were detected.

Therefore, no action is required.

In [11]:
df.duplicated().sum()

np.int64(0)

# Section 3: Categorical Data Validation

## Problem

Categorical variables should be validated before encoding to ensure consistency and detect any unexpected values, formatting issues, or data entry errors.

In [12]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print(categorical_columns)

['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


In [13]:
for column in categorical_columns:
    print("=" * 60)
    print(f"{column}")
    print(df[column].unique())
    print()

customerID
<StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7032, dtype: str

gender
<StringArray>
['Female', 'Male']
Length: 2, dtype: str

Partner
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

Dependents
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

PhoneService
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

MultipleLines
<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

InternetService
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

OnlineSecurity
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

OnlineBackup
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

DeviceProtection
<StringArray>
['No', 'Yes', 'No internet service']
L

In [15]:
categorical_columns = [
    col for col in df.select_dtypes(include="object").columns
    if col != "customerID"
]

In [16]:
categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique Values": [df[col].nunique() for col in categorical_columns],
    "Categories": [", ".join(map(str, sorted(df[col].unique()))) for col in categorical_columns]
})

categorical_summary

,Column,Unique Values,Categories
0,gender,2,"Female, Male"
1,Partner,2,"No, Yes"
2,Dependents,2,"No, Yes"
3,PhoneService,2,"No, Yes"
4,MultipleLines,3,"No, No phone service, Yes"
5,InternetService,3,"DSL, Fiber optic, No"
6,OnlineSecurity,3,"No, No internet service, Yes"
7,OnlineBackup,3,"No, No internet service, Yes"
8,DeviceProtection,3,"No, No internet service, Yes"
9,TechSupport,3,"No, No internet service, Yes"


In [18]:
# ===============================
# Final Validation
# ===============================

print("=" * 50)
print("FINAL DATA VALIDATION")
print("=" * 50)

print(f"Dataset Shape      : {df.shape}")
print(f"Missing Values     : {df.isnull().sum().sum()}")
print(f"Duplicate Rows     : {df.duplicated().sum()}")

print(f"Unique Customers   : {df['customerID'].nunique()}")
print(f"Memory Usage (MB)  : {round(df.memory_usage(deep=True).sum() / 1024**2, 2)}")

print("\nData Types")
print(df.dtypes.value_counts())

print("\nDataset is ready for Exploratory Data Analysis (EDA).")

FINAL DATA VALIDATION
Dataset Shape      : (7032, 21)
Missing Values     : 0
Duplicate Rows     : 0
Unique Customers   : 7032
Memory Usage (MB)  : 6.55

Data Types
str        17
int64       2
float64     2
Name: count, dtype: int64

Dataset is ready for Exploratory Data Analysis (EDA).


In [19]:
cleaning_log = pd.DataFrame({

    "Step":[
        1,
        2,
        3,
        4
    ],

    "Issue":[
        "Incorrect Data Type",
        "Blank Values",
        "Duplicate Records",
        "Categorical Validation"
    ],

    "Before":[
        "TotalCharges = object",
        11,
        0,
        "Unchecked"
    ],

    "After":[
        "TotalCharges = float64",
        0,
        0,
        "Validated"
    ],

    "Decision":[
        "Converted to Numeric",
        "Removed Records",
        "No Action Required",
        "No Action Required"
    ],

    "Status":[
        "✅ Fixed",
        "✅ Fixed",
        "✅ Passed",
        "✅ Passed"
    ]

})

cleaning_log

,Step,Issue,Before,After,Decision,Status
0,1,Incorrect Data Type,TotalCharges = object,TotalCharges = float64,Converted to Numeric,✅ Fixed
1,2,Blank Values,11,0,Removed Records,✅ Fixed
2,3,Duplicate Records,0,0,No Action Required,✅ Passed
3,4,Categorical Validation,Unchecked,Validated,No Action Required,✅ Passed


# Preprocessing Summary

The preprocessing phase has been completed successfully.

The following actions were performed:

- Corrected the data type of the `TotalCharges` column.
- Removed records containing blank values.
- Verified that no duplicate records exist.
- Validated all categorical variables.
- Confirmed that the dataset is clean and ready for exploratory data analysis.

In [20]:
df.to_csv("../data/cleaned/customer_churn_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
